# Exploração — API do Portal da Transparência

Objetivo: mapear endpoints relevantes, entender estrutura dos dados, confirmar o que podemos coletar.

Documentação: https://api.portaldatransparencia.gov.br/swagger-ui

**Endpoints que vamos explorar:**
- `/servidores` — servidores públicos municipais (para salário do prefeito)
- `/obras` — obras públicas federais por município
- `/auxilio-brasil` — Auxílio Brasil / Bolsa Família
- `/municipios` — catálogo de municípios (para confirmar código IBGE)

In [1]:
import os
import json
import requests
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

TOKEN = os.getenv("TRANSPARENCIA_API_TOKEN")
if not TOKEN:
    raise ValueError(
        "Token não encontrado no .env\n"
        "Obtenha gratuitamente em: https://api.portaldatransparencia.gov.br/swagger-ui\n"
        "Clique em 'Authorize' → preencha o e-mail → copie o token recebido por e-mail"
    )

BASE_URL = "https://api.portaldatransparencia.gov.br/api-de-dados"
HEADERS  = {"chave-api-dados": TOKEN, "Accept": "application/json"}

def get(endpoint: str, params: dict = None) -> requests.Response:
    url = f"{BASE_URL}/{endpoint}"
    r = requests.get(url, headers=HEADERS, params=params or {}, timeout=30)
    print(f"GET {endpoint} → {r.status_code}")
    return r

print("Token carregado:", TOKEN[:8] + "...")

Token carregado: a024ee90...


## 1. Catálogo de municípios do Acre
Primeiro passo: confirmar os códigos IBGE dos 22 municípios.

In [2]:
# Municípios do Acre
r = get("municipios", {"uf": "AC"})
r.json() if r.ok else print("ERRO:", r.text)

GET municipios → 403
ERRO: 


In [3]:
# Tentar sem UF — descobrir a estrutura do endpoint
r = get("municipios", {"pagina": 1})
if r.ok:
    dados = r.json()
    print("Tipo retornado:", type(dados))
    print("Primeiro item:", json.dumps(dados[0] if isinstance(dados, list) else dados, indent=2, ensure_ascii=False))
else:
    print("ERRO:", r.status_code, r.text[:500])

GET municipios → 403
ERRO: 403 


## 2. Servidores — salário do prefeito
Confirmar se o endpoint existe e qual é o filtro por município.

In [4]:
# Rio Branco — código IBGE 1200401
RIO_BRANCO = "1200401"

# Tentar buscar servidores de Rio Branco
r = get("servidores", {"codigoMunicipio": RIO_BRANCO, "pagina": 1})
if r.ok:
    dados = r.json()
    print(f"Registros retornados: {len(dados) if isinstance(dados, list) else 'N/A'}")
    if isinstance(dados, list) and dados:
        print("\nCampos disponíveis:")
        print(list(dados[0].keys()))
        print("\nPrimeiro registro:")
        print(json.dumps(dados[0], indent=2, ensure_ascii=False))
else:
    print("ERRO:", r.status_code, r.text[:1000])

GET servidores → 400
ERRO: 400 {"Erro na API":"Filtros mínimos:  Página (padrão = 1); Código Órgão Lotação (SIAPE) OU Código Órgão Exercício (SIAPE) OU CPF;"}


In [5]:
# Tentar variações do endpoint de servidores
for endpoint in ["servidores", "servidores/municipio", "remuneracao"]:
    r = requests.get(
        f"{BASE_URL}/{endpoint}",
        headers=HEADERS,
        params={"pagina": 1},
        timeout=10
    )
    print(f"  {endpoint}: {r.status_code}")

  servidores: 400


  servidores/municipio: 403


  remuneracao: 403


## 3. Obras públicas por município

In [6]:
# Obras em Rio Branco
r = get("obras", {"codigoMunicipio": RIO_BRANCO, "pagina": 1})
if r.ok:
    dados = r.json()
    print(f"Registros: {len(dados) if isinstance(dados, list) else 'N/A'}")
    if isinstance(dados, list) and dados:
        print("\nCampos:")
        print(list(dados[0].keys()))
        print("\nExemplo:")
        print(json.dumps(dados[0], indent=2, ensure_ascii=False))
else:
    print("ERRO:", r.status_code, r.text[:500])

GET obras → 403
ERRO: 403 


In [7]:
# Tentar com UF em vez de código de município
r = get("obras", {"uf": "AC", "pagina": 1})
if r.ok:
    dados = r.json()
    print(f"Obras no AC: {len(dados) if isinstance(dados, list) else dados}")
    if isinstance(dados, list) and dados:
        campos = list(dados[0].keys())
        print("Campos:", campos)
        # Checar se tem status de execução
        status_campos = [c for c in campos if "status" in c.lower() or "situa" in c.lower()]
        print("Campos de status:", status_campos)
else:
    print("ERRO:", r.status_code, r.text[:500])

GET obras → 403
ERRO: 403 


## 4. Transferências / FPM
Repasses federais para municípios — alternativa ao SICONFI.

In [8]:
# Transferências para Rio Branco
import datetime
ano_atual = datetime.date.today().year

r = get("transferencias-constitucionais", {
    "codigoMunicipio": RIO_BRANCO,
    "ano": ano_atual - 1,
    "pagina": 1
})
if r.ok:
    dados = r.json()
    print(f"Registros: {len(dados) if isinstance(dados, list) else dados}")
    if isinstance(dados, list) and dados:
        print("Campos:", list(dados[0].keys()))
        print(json.dumps(dados[0], indent=2, ensure_ascii=False))
else:
    print("ERRO:", r.status_code, r.text[:500])

GET transferencias-constitucionais → 403
ERRO: 403 


## 5. Bolsa Família / Auxílio Brasil
Verificar se existe endpoint REST além do BD+.

In [9]:
# Benefícios por município
r = get("bolsa-familia-disponibilidade", {"codigoMunicipio": RIO_BRANCO, "pagina": 1})
if r.ok:
    dados = r.json()
    print(f"Registros: {len(dados) if isinstance(dados, list) else dados}")
    if isinstance(dados, list) and dados:
        print(json.dumps(dados[0], indent=2, ensure_ascii=False))
else:
    print("ERRO:", r.status_code)
    # Tentar variações do nome
    for ep in ["bolsa-familia", "auxilio-brasil", "beneficios-sociais"]:
        r2 = requests.get(f"{BASE_URL}/{ep}", headers=HEADERS, params={"pagina": 1}, timeout=10)
        print(f"  {ep}: {r2.status_code}")

GET bolsa-familia-disponibilidade → 403
ERRO: 403


  bolsa-familia: 403


  auxilio-brasil: 403


  beneficios-sociais: 403


## 6. Resumo — o que encontramos
Documentar o que a API entrega vs o que precisamos buscar de outra forma.

In [10]:
# Preencher após exploração
resultado = {
    "municipios":            {"endpoint": "municipios",          "status": None, "campos_chave": []},
    "salario_prefeito":      {"endpoint": "servidores",           "status": None, "campos_chave": []},
    "obras":                 {"endpoint": "obras",                "status": None, "campos_chave": []},
    "transferencias":        {"endpoint": "transf-constitucionais","status": None, "campos_chave": []},
    "bolsa_familia_api":     {"endpoint": "bolsa-familia-*",      "status": None, "campos_chave": []},
}

print("\n=== PREENCHER MANUALMENTE APÓS CÉLULAS ACIMA ===")
print(json.dumps(resultado, indent=2, ensure_ascii=False))


=== PREENCHER MANUALMENTE APÓS CÉLULAS ACIMA ===
{
  "municipios": {
    "endpoint": "municipios",
    "status": null,
    "campos_chave": []
  },
  "salario_prefeito": {
    "endpoint": "servidores",
    "status": null,
    "campos_chave": []
  },
  "obras": {
    "endpoint": "obras",
    "status": null,
    "campos_chave": []
  },
  "transferencias": {
    "endpoint": "transf-constitucionais",
    "status": null,
    "campos_chave": []
  },
  "bolsa_familia_api": {
    "endpoint": "bolsa-familia-*",
    "status": null,
    "campos_chave": []
  }
}
